# 01 · 快速上手：跑通第一个 LoRA 微调

本教程带你走完一次完整微调流程：**配置 → 数据 → 训练 → 产物**。

| 环节 | 对应模块 |
|---|---|
| 配置 | `llm_finetuning.Config` |
| 数据 | `data.processor` / `data.cleaner` |
| 训练 | `FinetuningEngine` |
| 产物 | `export.exporter` |

> 建议先 `pip install -e ".[minimal]"`。没有 GPU 也能读完本教程 ——
> 需要 GPU 的单元会自动跳过并给出提示，配置与数据部分始终可运行。

In [ ]:
# 环境自检
import importlib
import sys

print(f"Python {sys.version.split()[0]}")

for pkg in ("torch", "transformers", "datasets", "peft", "accelerate"):
    try:
        mod = importlib.import_module(pkg)
        print(f"  OK  {pkg:<12} {getattr(mod, '__version__', 'unknown')}")
    except ImportError:
        print(f"  --  {pkg:<12} 未安装")

try:
    import torch

    print(f"\nCUDA 可用: {torch.cuda.is_available()}")
    if torch.cuda.is_available():
        props = torch.cuda.get_device_properties(0)
        print(f"GPU: {props.name}，显存 {props.total_memory / 1024 ** 3:.1f} GB")
except ImportError:
    print("\n未安装 torch：训练单元将跳过，配置与数据部分仍可运行")

## 1. 构建配置

`Config` 是项目入口。指定 `finetuning_method` 后会自动填好该方法惯用的默认超参
（LoRA 学习率 2e-4、QLoRA 默认 4-bit 等），只覆盖你关心的字段即可。

In [ ]:
import sys

sys.path.insert(0, "..")  # notebooks/ 下运行时指向仓库根

from llm_finetuning import Config, FinetuningMethod

config = Config(
    model_name_or_path="Qwen/Qwen2.5-0.5B-Instruct",  # 演示用小模型
    finetuning_method=FinetuningMethod.LORA,
    output_dir="./outputs/quickstart",
)

config.training.num_train_epochs = 1
config.training.per_device_train_batch_size = 4
config.training.gradient_accumulation_steps = 4
config.training.save_strategy = "no"
config.data.max_seq_length = 512

import json

print(json.dumps(config.to_dict(), ensure_ascii=False, indent=2)[:800])

## 2. 准备数据

项目接受 `instruction / input / output` 三字段的 JSONL。
下面生成一份样例数据（真实场景替换为自己的文件），再清洗一次。

`DataCleaner` 按 `text_field` 指定的字段做校验与去重，这里用 `output`
—— 因为在指令微调里，**回答的质量比问题更值得先检查**。

In [ ]:
from llm_finetuning import DataCleaner, DatasetLoader, DataConfig
from llm_finetuning import create_sample_instruction_dataset

dataset_path = create_sample_instruction_dataset(
    output_path="./outputs/sample_instructions.jsonl",
    num_samples=50,
)
samples = DatasetLoader(DataConfig()).load_from_jsonl(dataset_path)

print(f"样本数: {len(samples)}")
print(samples[0])

cleaner = DataCleaner(text_field="output", min_length=5, max_length=1024, fuzzy_dedup=True)
clean_samples, report = cleaner.clean(samples)

print(f"\n清洗后: {len(clean_samples)}")
print(report.summary())

## 3. 训练

`FinetuningEngine` 屏蔽了各微调方法的差异：换 `finetuning_method` 即可切换
LoRA / QLoRA / 全参，其余代码不变。

`load_data()` 返回 `(train, eval)`；`train()` 接收这两个数据集。
首次运行会从 HuggingFace 下载权重，网络受限时请先设置 `HF_ENDPOINT` 镜像。

In [ ]:
from llm_finetuning import FinetuningEngine

engine = FinetuningEngine(config)

try:
    import torch  # noqa: F401

    train_dataset, eval_dataset = engine.load_data(dataset_path=dataset_path)
    print(f"train={train_dataset} eval={eval_dataset}")

    # 真正开始训练（小模型 1 个 epoch，几分钟内可完成）：
    # engine.train(train_dataset, eval_dataset)
    print("已就绪；取消注释 engine.train(...) 即可开始训练")
except ImportError:
    print("未安装 torch/transformers，跳过训练环节")
    print("安装: pip install -e '.[minimal]'")
except Exception as exc:  # 网络/显存等问题不阻塞教程阅读
    print(f"训练环节未执行: {type(exc).__name__}: {exc}")

## 4. 接下来

- 压显存 / 选方法 → `02_lora_qlora.ipynb`
- 提数据质量 → `03_data_preparation.ipynb`
- 调超参 → `05_hyperparameter_search.ipynb`
- 评估与上线 → `04_evaluate_export_deploy.ipynb`